### 1. Set up env and access api keys to test


In [ ]:
from dotenv import load_dotenv
import json
from datetime import datetime
import os
import re
import sys
import pandas as pd
from openai import OpenAI
import gradio as gr
import subprocess
from IPython.display import Markdown, display

load_dotenv()

In [ ]:
openrouter_api_key = os.environ.get("OPENROUTER_API_KEY")
alpaca_key = os.environ.get("ALPACA_API_KEY")
alpaca_secret = os.environ.get("ALPACA_SECRET_KEY")
alpaca_endpoint = os.environ.get("ALPACA_PAPER_ENDPOINT")

In [ ]:
# Connect to client libraries

openai = OpenAI()

openrouter_url = "https://openrouter.ai/api/v1"

openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)


### 2. Set up models dict


In [ ]:
JUDGE_MODEL = "gpt-5"
JUDGE_CLIENT = openai

models = [
    "gpt-5.5",
    "moonshotai/kimi-k2.6",
    "openai/gpt-oss-120b:free",
    "openai/gpt-oss-20b:free",
    "deepseek/deepseek-v4-pro",
    "anthropic/claude-sonnet-4.6",
]

clients = {
    "gpt-5.5": openai,
    "moonshotai/kimi-k2.6": openrouter,
    "openai/gpt-oss-120b:free": openrouter,
    "openai/gpt-oss-20b:free": openrouter,
    "deepseek/deepseek-v4-pro": openrouter,
    "anthropic/claude-sonnet-4.6": openrouter,
}


In [ ]:
REASONING_MODELS = {
    "moonshotai/kimi-k2.6",
    "openai/gpt-oss-120b:free",
    "openai/gpt-oss-20b:free",
    "deepseek/deepseek-v4-pro",
}

### 3. Write the master code-generation prompt


In [ ]:
BUY_PRICE = 300
SELL_PRICE = 310

In [ ]:
MASTER_PROMPT = f"""Write a complete Python script for Alpaca paper trading only. Return Python code only with no markdown fences and no commentary.\n
\n
Requirements:\n
- Trade only AAPL.\n
- Use real Alpaca market data via the latest trade endpoint specifically.\n
- Fetch the latest AAPL trade price.\n
- Read credentials from .env using these exact variable names: OPENROUTER_API_KEY, ALPACA_API_KEY, ALPACA_SECRET_KEY, ALPACA_PAPER_ENDPOINT (with the value https://paper-api.alpaca.markets/v2).\n
- Use Alpaca paper trading only. Never use live trading endpoints or live trading credentials.\n
- Submit at most one paper order per run.\n
- If the latest AAPL price is below {BUY_PRICE}, check Alpaca account buying power and buy exactly 1 share only if buying power is sufficient. Otherwise skip and print the reason.\n
- If the latest AAPL price is above {SELL_PRICE}, check the current AAPL position and sell exactly 1 share only if at least 1 share is already held. Otherwise skip and print the reason. Never short.\n
- If the latest AAPL price is between {BUY_PRICE} and {SELL_PRICE} inclusive, do nothing and print the reason.\n
- If the US market is closed, still proceed using the last available trade price.\n
- Print the latest trade price, decision made, precondition checks, whether an order was submitted, and the order response or error message.\n
- Handle missing credentials gracefully.\n
- Handle Alpaca API errors gracefully.\n
- Avoid infinite loops or repeated polling.\n
- Avoid hardcoded API keys.\n
- Avoid portfolio-wide logic.\n
- Avoid any ticker except AAPL.\n
"""


### 4. Generate code from one model


In [ ]:
def _strip_code_fences(text):
    text = text.strip()
    if not text.startswith("```"):
        return text

    lines = text.splitlines()
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].strip() == "```":
        lines = lines[:-1]
    return "\n".join(lines).strip()


def generate_code(model_id, prompt):
    """Generate source code for one selected model.

    Args:
        model_id: Model identifier from the notebook's model list.
        prompt: Master prompt sent as the user message.

    Returns:
        Plain Python source with markdown fences removed.

    Raises:
        gr.Error: If the model is unknown, the API call fails, or the response is empty.
    """
    client = clients.get(model_id)
    if client is None:
        raise gr.Error(f"Unknown model: {model_id}")

    kwargs = {
        "model": model_id,
        "messages": [{"role": "user", "content": prompt}],
    }

    if model_id in REASONING_MODELS:
        kwargs["extra_body"] = {"reasoning": {"effort": "high"}}

    try:
        response = client.chat.completions.create(**kwargs)
        content = response.choices[0].message.content or ""
    except Exception as exc:
        raise gr.Error(f"Code generation failed for {model_id}: {exc}") from exc

    if not content.strip():
        raise gr.Error(f"Model returned empty output for {model_id}.")

    return _strip_code_fences(content)


In [ ]:
# generated_model_code = generate_code(
#     model_id="deepseek/deepseek-v4-pro", prompt=MASTER_PROMPT
# )

### 5. Save generated scripts


In [ ]:
def _safe_model_label(model_label):
    cleaned = [char.lower() if char.isalnum() else "_" for char in model_label.strip()]
    # isalnum checks with a string contains only alphanumeric characters, basically it removes everything other than alphabets and numbers
    safe_label = "".join(cleaned).strip("_")
    return safe_label or "model"


def save_generated_script(model_label, code, output_dir="outputs"):
    """Save generated model code to a predictable Python file.

    Args:
        model_label: Display label used to build the output filename.
        code: Generated Python source to save.
        output_dir: Folder where generated scripts are written.

    Returns:
        Relative path to the saved script.

    Raises:
        gr.Error: If the generated code is empty.
    """
    if not code.strip():
        raise gr.Error("Generated code is empty; nothing to save.")

    os.makedirs(output_dir, exist_ok=True)
    filename = f"generated_{_safe_model_label(model_label)}.py"
    file_path = os.path.join(output_dir, filename)

    with open(file_path, "w", encoding="utf-8") as handle:
        handle.write(code)

    return file_path


### 7. Run generated scripts manually from Gradio


In [ ]:
def run_generated_script(script_path, timeout=120):
    """Run a saved Python script with the notebook's interpreter.

    Args:
        script_path: Relative or absolute path to the generated script.
        timeout: Max seconds to allow the subprocess to run.

    Returns:
        Dict with stdout (normal output by the script, like print statements), stderr (error messages from the script, like ValueError or NameError etc.), exit_code (0 means success), and timed_out keys.

    Raises:
        gr.Error: If the script path does not exist.
    """
    if not os.path.exists(script_path):
        raise gr.Error(f"Script not found: {script_path}")

    try:
        result = subprocess.run(
            [sys.executable, script_path],
            cwd=os.getcwd(),
            capture_output=True,
            text=True,  # give the normal output of the script (stdout) and stderr as normal python STRINGS.
            timeout=timeout,
        )
    except subprocess.TimeoutExpired as exc:
        return {
            "stdout": exc.stdout or "",
            "stderr": exc.stderr or "",
            "exit_code": None,
            "timed_out": True,
        }

    return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "exit_code": result.returncode,
        "timed_out": False,
    }


### 8. Load leaderboard data


In [ ]:
LEADERBOARD_COLUMNS = [
    "timestamp",
    "model_label",
    "model_id",
    "final_score",
    "raw_judge_score",
    "status",
    "cap_reason",
    "main_issue",
]


def score_status(final_score):
    """Map a final score to the leaderboard status label.

    Args:
        final_score: Integer score after hard caps.

    Returns:
        Status string used in the leaderboard.
    """
    if final_score == 0:
        return "Execution Failed"
    if final_score >= 80:
        return "Pass"
    if final_score >= 60:
        return "Partial"
    return "Fail"


def append_leaderboard_row(
    model_id, cap_result, judge_result, csv_path="outputs/leaderboard.csv"
):
    """Append one evaluation row to the leaderboard CSV.

    Args:
        model_id: Evaluated model identifier.
        cap_result: Dict returned by apply_judge_caps().
        judge_result: Dict returned by judge_model_output().
        csv_path: Path to the leaderboard CSV file.

    Returns:
        DataFrame with the updated leaderboard contents.
    """
    row = pd.DataFrame(
        [
            {
                "timestamp": datetime.now().isoformat(timespec="seconds"),
                "model_label": model_id,
                "model_id": model_id,
                "final_score": cap_result["final_score"],
                "raw_judge_score": cap_result["raw_judge_score"],
                "status": score_status(cap_result["final_score"]),
                "cap_reason": cap_result["cap_reason"],
                "main_issue": (judge_result.get("issues") or ["—"])[0],
            }
        ]
    ).reindex(
        columns=LEADERBOARD_COLUMNS
    )  # reindex means make sure this DataFrame has exactly these columns (LEADERBOARD_COLUMNS), in exactly this order. (a guardrail)

    write_header = not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0
    row.to_csv(csv_path, mode="a", header=write_header, index=False)
    return load_leaderboard(csv_path)


def load_leaderboard(csv_path="outputs/leaderboard.csv"):
    """Load leaderboard rows from CSV or return an empty table.

    Args:
        csv_path: Path to the leaderboard CSV file.

    Returns:
        DataFrame with the standard leaderboard columns, sorted by final_score descending.
    """
    if not os.path.exists(csv_path) or os.path.getsize(csv_path) == 0:
        return pd.DataFrame(columns=LEADERBOARD_COLUMNS)

    try:
        leaderboard = pd.read_csv(csv_path)
    except pd.errors.EmptyDataError:
        return pd.DataFrame(columns=LEADERBOARD_COLUMNS)

    leaderboard = leaderboard.reindex(columns=LEADERBOARD_COLUMNS)
    return leaderboard.sort_values(
        "final_score", ascending=False, na_position="last"
    )  # na means NaN (missing values), na_position="last" means put the rows with missing values at the bottom of the table.


In [ ]:
load_leaderboard()

### 9. Build the Gradio UI


In [ ]:
def build_arena_ui():
    """Build the TradeCode Arena UI and bind the sequential button handlers."""

    def on_generate(model_id):
        code = generate_code(model_id, MASTER_PROMPT)
        script_path = save_generated_script(model_id, code)
        empty_result = {
            "stdout": "",
            "stderr": "",
            "exit_code": None,
            "timed_out": False,
        }
        return (
            code,
            code,
            script_path,
            None,
            empty_result,
            {},
            {},
            None,
            gr.update(interactive=True),
            gr.update(interactive=False),
        )

    def on_execute(script_path):
        result = run_generated_script(script_path)
        return result, result, gr.update(interactive=True)

    def on_evaluate(model_id, code, execution_result, judge_model):
        if execution_result is None:
            raise gr.Error("Execute the generated code before evaluation.")
        judge_result = judge_model_output(code, execution_result, judge_model)
        cap_result = apply_judge_caps(code, execution_result, judge_result)
        leaderboard_df = append_leaderboard_row(model_id, cap_result, judge_result)
        return (
            {
                "raw_judge_score": cap_result["raw_judge_score"],
                "cap_reason": cap_result["cap_reason"],
                "triggered_caps": cap_result["triggered_caps"],
            },
            judge_result,
            cap_result["final_score"],
            leaderboard_df,
        )

    with gr.Blocks(title="TradeCode Arena") as app:
        app_state = {
            "generated_code_state": gr.State(
                ""
            ),  # plain Python code generated as string
            "script_path_state": gr.State(""),  # path to generated script
            "execution_result_state": gr.State(
                None
            ),  # dict with stdout, stderr, exit_code, timed_out
        }
        gr.Markdown("# TradeCode Arena")
        with gr.Row():
            model_id = gr.Dropdown(
                choices=models,
                value=models[0],
                label="Model",
                interactive=True,
            )
            judge_model = gr.Textbox(value=JUDGE_MODEL, label="Judge Model")
        with gr.Row():
            generate_btn = gr.Button("Generate Code")
            execute_btn = gr.Button("Execute Code", interactive=False)
            evaluate_btn = gr.Button("Evaluate Model", interactive=False)
        generated_code = gr.Code(
            label="Generated Code", language="python", interactive=False
        )
        execution_output = gr.JSON(
            label="Execution Result",
            value={"stdout": "", "stderr": "", "exit_code": None, "timed_out": False},
        )
        with gr.Row():
            static_checks = gr.JSON(label="Cap Result", value={})
            judge_feedback = gr.JSON(label="Judge Feedback", value={})
        final_score = gr.Number(label="Final Score", value=None)
        leaderboard = gr.Dataframe(label="Leaderboard", value=load_leaderboard())

        generate_btn.click(
            fn=on_generate,
            inputs=[model_id],
            outputs=[
                generated_code,
                app_state["generated_code_state"],
                app_state["script_path_state"],
                app_state["execution_result_state"],
                execution_output,
                static_checks,
                judge_feedback,
                final_score,
                execute_btn,
                evaluate_btn,
            ],
        )
        execute_btn.click(
            fn=on_execute,
            inputs=[app_state["script_path_state"]],
            outputs=[
                execution_output,
                app_state["execution_result_state"],
                evaluate_btn,
            ],
        )
        evaluate_btn.click(
            fn=on_evaluate,
            inputs=[
                model_id,
                app_state["generated_code_state"],
                app_state["execution_result_state"],
                judge_model,
            ],
            outputs=[static_checks, judge_feedback, final_score, leaderboard],
        )

    # components = {
    #     **app_state,
    #     "model_id": model_id,
    #     "judge_model": judge_model,
    #     "generate_btn": generate_btn,
    #     "execute_btn": execute_btn,
    #     "evaluate_btn": evaluate_btn,
    #     "generated_code": generated_code,
    #     "execution_output": execution_output,
    #     "static_checks": static_checks,
    #     "judge_feedback": judge_feedback,
    #     "final_score": final_score,
    #     "leaderboard": leaderboard,
    # }
    return app


### 10. Wire the Gradio handlers


In [ ]:
# Event wiring moved into build_arena_ui() because Gradio click handlers
# must be registered inside the active gr.Blocks context.


### Phase 10. Judge LLM evaluation (merged with phase 8 & 9)


In [ ]:
JUDGE_RUBRIC = """Score the generated trading code out of 100 using this rubric:
Correct Alpaca SDK / API structure: 20
Uses paper trading only: 20
Uses .env credentials correctly: 15
Follows AAPL buy/sell strategy with preconditions: 15
Uses real market data (latest trade): 10
Risk control: at most 1 order, qty 1, no shorting: 10
Handles errors / missing credentials: 5
Code readability: 5
Return JSON only with keys raw_score, strengths, issues, summary."""


def judge_model_output(code, execution_result, judge_model):
    """Judge generated trading code with the configured LLM and parse its JSON output.

    Args:
        code: Generated Python source.
        execution_result: Dict with stdout, stderr, exit_code, and timed_out.
        judge_model: OpenRouter model ID used for judging.

    Returns:
        Dict with raw_score, strengths, issues, and summary.
    """
    prompt = f"""{JUDGE_RUBRIC}

Master prompt:
{MASTER_PROMPT}

Generated code:
{code}

Execution result JSON:
{json.dumps(execution_result, indent=2)}
"""
    fallback = {
        "raw_score": 0,
        "strengths": [],
        "issues": ["Judge returned invalid JSON."],
        "summary": "Judge evaluation failed.",
    }
    try:
        response = openrouter.chat.completions.create(
            model=judge_model,
            messages=[{"role": "user", "content": prompt}],
        )
        content = (response.choices[0].message.content or "").strip()
    except Exception as exc:
        fallback["issues"] = [f"Judge request failed: {exc}"]
        fallback["summary"] = "Judge request failed."
        return fallback

    if content.startswith("```"):
        content = _strip_code_fences(content)
    if not content.startswith("{"):
        match = re.search(r"\{.*\}", content, re.DOTALL)
        content = match.group(0) if match else content
    try:
        result = json.loads(content)
    except json.JSONDecodeError:
        return fallback

    return {
        "raw_score": int(result.get("raw_score", 0)),
        "strengths": result.get("strengths", []),
        "issues": result.get("issues", []),
        "summary": result.get("summary", ""),
    }


### Phase 10B. Judge-based hard caps


In [ ]:
JUDGE_CAP_RULES = [
    # Each tuple is: (maximum_allowed_score, reason, check_function)
    # The lambda checks code/issues and returns True if this cap should apply.
    (
        20,
        "Live trading endpoint detected",
        lambda code, issues: "api.alpaca.markets" in code.lower()
        and "paper-api.alpaca.markets" not in code.lower(),
    ),
    (
        20,
        "Paper trading not used",
        lambda code, issues: "paper trading" in issues
        and any(
            word in issues for word in ["missing", "not use", "not used", "incorrect"]
        ),
    ),
    (
        40,
        "Real market data not fetched",
        lambda code, issues: any(
            phrase in issues
            for phrase in ["market data", "real market data", "market data missing"]
        ),
    ),
    (
        60,
        "No Alpaca order submission found",
        lambda code, issues: any(
            phrase in issues
            for phrase in ["no order", "order submission missing", "does not submit"]
        ),
    ),
    # (
    #     60,
    #     "Ticker other than AAPL detected",
    #     # re.findall returns a list of ticker-like strings.
    #     # set(...) - {"AAPL"} leaves only tickers that are NOT AAPL.
    #     # bool(...) is True if that leftover set is not empty.
    #     lambda code, issues: bool(
    #         set(re.findall(r"['\"]([A-Z]{1,5})['\"]", code)) - {"AAPL"}
    #     ),
    # ), # this is removed due to high possibility of false positives. 'SELL' gets considered as a ticker.
    (
        60,
        "Possible short selling without position check",
        lambda code, issues: "short" in issues,
    ),
    # (
    #     70,
    #     "Possible multiple orders per run",
    #     # re.findall returns all matches.
    #     # len(...) > 1 means it found more than one possible order call.
    #     lambda code, issues: "more than one order" in issues
    #     or len(re.findall(r"submit_order\s*\(|requests\.post\(", code.lower())) > 1,
    # ), # this hard limit is removed as 'submit_order' can appear in one if block and in another elif or if block again.
    (
        70,
        "Buy logic missing buying-power check",
        lambda code, issues: "buying power" in issues,
    ),
    (
        70,
        "Possible hardcoded API key or secret",
        # re.search returns a match object if found, or None if not found.
        # bool(...) converts that into True or False.
        lambda code, issues: bool(
            re.search(
                r"(api_key|secret_key)\s*=\s*['\"][^'\"]+['\"]", code, re.IGNORECASE
            )
        ),
    ),
]


def apply_judge_caps(code, execution_result, judge_result):
    """Apply hard caps using execution results, judge findings, and direct code checks.

    Args:
        code: Generated Python source.
        execution_result: Runner result dict.
        judge_result: Output from judge_model_output().

    Returns:
        Dict with final_score, raw_judge_score, cap_reason, and triggered_caps.
    """
    # Get the judge's original score. If missing, use 0.
    raw_score = int(judge_result.get("raw_score", 0))

    # If the generated code crashed or timed out, score is automatically 0.
    # exit_code == 0 means success. Anything else means failure.
    if execution_result.get("timed_out") or execution_result.get("exit_code") != 0:
        return {
            "final_score": 0,
            "raw_judge_score": raw_score,
            "cap_reason": "Execution failed",
            "triggered_caps": [0],
        }

    # Judge issues start as a list, then become one lowercase string for easy text checks.
    issues = " ".join(judge_result.get("issues", [])).lower()

    # Run every cap rule.
    # predicate is the variable holding the lambda function from each tuple.
    # predicate(code, issues) calls that lambda and returns True or False. the meaning of predicate in programming terms is just a function that returns true or false.
    triggered = [
        (cap, reason)
        for cap, reason, predicate in JUDGE_CAP_RULES
        if predicate(code, issues)
    ]

    # If no cap rules triggered, keep the judge's original score.
    if not triggered:
        return {
            "final_score": raw_score,
            "raw_judge_score": raw_score,
            "cap_reason": "—",
            "triggered_caps": [],
        }

    # More than one cap can trigger.
    # Pick the lowest cap because it is the strictest one.
    cap, reason = min(triggered, key=lambda item: item[0])

    return {
        # The cap can only lower the score, never raise it.
        "final_score": min(raw_score, cap),
        "raw_judge_score": raw_score,
        "cap_reason": reason,
        # Store all triggered cap numbers, not only the strictest one.
        "triggered_caps": [item[0] for item in triggered],
    }


In [ ]:
app = build_arena_ui()
app.launch(inbrowser=True)